<a href="https://colab.research.google.com/github/safdarmarwat/colab/blob/main/training_tuning01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets sentencepiece accelerate

In [2]:
import re
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments, DataCollatorForSeq2Seq

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [3]:
# Loading the tweet_eval sentiment dataset
dataset = load_dataset("tweet_eval", "sentiment", split="train[:100]")
print(f"Loaded {len(dataset)} samples.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loaded 100 samples.


In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE) # Remove URLs
    text = re.sub(r'\@\w+|\#','', text) # Remove mentions and hashtags
    text = re.sub(r'[^\w\s]', '', text) # Remove special characters
    return text.strip()

# Show Before vs After
sample_text = dataset[0]['text']
print(f"BEFORE: {sample_text}")
print(f"AFTER:  {clean_text(sample_text)}")

BEFORE: "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"
AFTER:  qt  in the original draft of the 7th book remus lupin survived the battle of hogwarts happybirthdayremuslupin


In [7]:
def generate_response(text, current_model):
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
    outputs = current_model.generate(input_ids, max_new_tokens=10)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Show BEFORE Fine-Tuning output
test_input = formatted_dataset[0]['input_text']
print(f"Input: {test_input}")
print(f"Pre-trained output: {generate_response(test_input, model)}")

Input: generate a polite response: qt  in the original draft of the 7th book remus lupin survived the battle of hogwarts happybirthdayremuslupin
Pre-trained output: survived the battle of hogwarts


In [9]:
model_id = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_id)
model = T5ForConditionalGeneration.from_pretrained(model_id).to(device)

def tokenize_batch(examples):
    # Tokenize the input text
    model_inputs = tokenizer(examples["input_text"], max_length=128, truncation=True, padding="max_length")

    # Tokenize the target text (the "apology", "appreciation", etc.)
    # Note: we use 'text_target' to tell the tokenizer these are labels
    labels = tokenizer(text_target=examples["target_text"], max_length=10, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# IMPORTANT: We use remove_columns to get rid of the original integer 'label'
# so the Trainer doesn't get confused.
tokenized_dataset = formatted_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=formatted_dataset.column_names
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [10]:
def generate_response(text, current_model):
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
    outputs = current_model.generate(input_ids, max_new_tokens=10)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Show BEFORE Fine-Tuning output
test_input = formatted_dataset[0]['input_text']
print(f"Input: {test_input}")
print(f"Pre-trained output: {generate_response(test_input, model)}")

Input: generate a polite response: qt  in the original draft of the 7th book remus lupin survived the battle of hogwarts happybirthdayremuslupin
Pre-trained output: survived the battle of hogwarts


In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    logging_steps=10,
    learning_rate=5e-4,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

trainer.train()

Step,Training Loss
10,5.284895
20,0.821045
30,0.222626
40,0.139080
50,0.108356
60,0.102946


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=65, training_loss=1.0350394652439998, metrics={'train_runtime': 278.2337, 'train_samples_per_second': 1.797, 'train_steps_per_second': 0.234, 'total_flos': 16917725184000.0, 'train_loss': 1.0350394652439998, 'epoch': 5.0})

In [12]:
print("--- FINAL TEST ON 5 EXAMPLES ---")
for i in range(5):
    raw_text = dataset[i]['text']
    cleaned_input = f"generate a polite response: {clean_text(raw_text)}"
    expected = map_sentiment(dataset[i]['label'])

    # We use the now-trained 'model' object
    fine_tuned_output = generate_response(cleaned_input, model)

    print(f"\nExample {i+1}:")
    print(f"Original Text: {raw_text}")
    print(f"Expected Tone: {expected}")
    print(f"Model Output:  {fine_tuned_output}")

--- FINAL TEST ON 5 EXAMPLES ---

Example 1:
Original Text: "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"
Expected Tone: appreciation
Model Output:  clarification

Example 2:
Original Text: "Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"
Expected Tone: clarification
Model Output:  appreciation

Example 3:
Original Text: Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.
Expected Tone: clarification
Model Output:  clarification

Example 4:
Original Text: Chase Headley's RBI double in the 8th inning off David Price snapped a Yankees streak of 33 consecutive scoreless innings against Blue Jays
Expected Tone: clarification
Model Output:  appreciation

Example 5:
Original Text: @user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017"
Expected Tone: app